In [5]:
import sys
import pandas as pd
import numpy as np

In [6]:
# preprocessing / integration
from benchmark_sigs.preprocess.integrate import integrate_data
from benchmark_sigs.preprocess.RNA import preprocess_rna_for_simulation, select_genes_with_expr_filter

# alteration simulation / utilities
from benchmark_sigs.simulate.alterations import (
    simulate_X,
    split_simulated_blocks,
    gistic_to_amp_del_binary,
)

# expression simulation
from benchmark_sigs.simulate.rna import simulate_rna_with_signatures

# saving
from benchmark_sigs.io.writers import save_simulation_outputs

In [11]:
'''
AML
'''



mutation_matched, cna_matched, fusion_matched, clinical_subtypes, clinical_matched, rna_matched= integrate_data(
    mut_path="/home/alanah/Downloads/AML/laml_tcga_pan_can_atlas_2018(1)/laml_tcga_pan_can_atlas_2018/data_mutations.txt",
    cna_path="/home/alanah/Downloads/AML/laml_tcga_pan_can_atlas_2018(1)/laml_tcga_pan_can_atlas_2018/data_cna.txt",
    fusion_info_path="/home/alanah/Downloads/AML/laml_tcga_pan_can_atlas_2018(1)/laml_tcga_pan_can_atlas_2018/data_sv.txt",
    patient_path="/home/alanah/Downloads/AML/laml_tcga_pan_can_atlas_2018(1)/laml_tcga_pan_can_atlas_2018/data_clinical_patient.txt",
    sample_path= "/home/alanah/Downloads/AML/laml_tcga_pan_can_atlas_2018(1)/laml_tcga_pan_can_atlas_2018/data_clinical_sample.txt",
    rna_path = 'exp_df_aml.csv',
    study='TCGA',
    disease="AML",
    cna_process=True,
    cna_top_n=200,
    min_subtype_n=6,
    mut_freq_thresh=0.02,
    fusion_freq_thresh=0.02,
    uncertain_top_k=100
)



sim_output = simulate_X(
    mut=mutation_matched,
    subtype=clinical_subtypes,
    n_samples=mutation_matched.shape[0],
    fusion=fusion_matched,
    cna=cna_matched,
    clinical=clinical_matched,
    seed=44,
    k_neighbors=5
)

#cna_matched.columns = cna_matched.columns.str.replace("_CNA_CNA", "_CNA", regex=True)


mut_sim, fus_sim, cna_sim_gistic, clin_sim, subtype_sim = split_simulated_blocks(sim_output)
cna_sim_binary = gistic_to_amp_del_binary(cna_sim_gistic)


cna_bin = gistic_to_amp_del_binary(cna_matched)
alts_real = pd.concat(
    [mutation_matched, fusion_matched, cna_bin],
    axis=1, join='inner'
)
constant_cols = alts_real.columns[alts_real.nunique(dropna=False) <= 1]
alts_real = alts_real.drop(columns=constant_cols)

alts_cna_gistic = pd.concat([mut_sim, fus_sim, cna_sim_gistic], axis=1, join='inner')
constant_cols = alts_cna_gistic.columns[alts_cna_gistic.nunique(dropna=False) <= 1]
alts_cna_gistic = alts_cna_gistic.drop(columns=constant_cols)

alts_bin = pd.concat([mut_sim, fus_sim, cna_sim_binary], axis=1, join='inner')
constant_cols = alts_bin.columns[alts_bin.nunique(dropna=False) <= 1]
alts_bin = alts_bin.drop(columns=constant_cols)


out = select_genes_with_expr_filter(
    rna_df=rna_matched,
    alterations_df=alts_real,
    target_total=10000,
    min_cpm=1.0,
    min_prop_samples=0.20,
    use_mad=False,    # set True if you prefer MAD over variance
    verbose=True
)

genes_to_keep = out["genes_to_keep"]

# Step 6: Final RNA matrix
rna_real = rna_matched[genes_to_keep]
rna_real = rna_real.loc[:, ~rna_real.columns.duplicated()]
# Step 6.5: Drop or fill NaNs before scaling
rna_real = rna_real.fillna(0)  # or rna_real.dropna(axis=1), but fillna is safer here
common_idx_rna = rna_real.index.intersection(mutation_matched.index)
rna_real = rna_real.loc[common_idx_rna]

rna_real_scaled, scale_used = preprocess_rna_for_simulation(rna_real, strategy="auto")
clinical_subtypes = clinical_subtypes.squeeze()


expr_sim, true_signatures, background_mu = simulate_rna_with_signatures(
    rna_df=rna_real_scaled,
    alteration_df=alts_bin,
    altertaions_real=alts_real,
    cna_df=cna_sim_gistic,
    n_samples=alts_bin.shape[0],
    subtype=clinical_subtypes,
    n_genes_to_sim=10000,
    seed=44,
    deseq2_summary_path="/home/alanah/Documents/AML/deseq2_alt_summary.csv",
    cap_k=8,
    global_cap=12,
)


save_simulation_outputs(
    rna_real=rna_real_scaled,
    expr_sim=expr_sim,
    alterations_df=alts_bin,
    gistic_alt=alts_cna_gistic,
    true_signatures=true_signatures,
    alt_real=alts_real,
    output_dir="/home/alanah/Documents",
    suffix="AML",
    compress=False
)


[INFO] Clinical: mapped 200 samples


/home/alanah/repo_test/Expression-Signature-Discovery/src/benchmark_sigs/preprocess/integrate.py:63: DtypeWarning: Columns (33,34,35,36,45) have mixed types. Specify dtype option on import or set low_memory=False.
  maf = pd.read_csv(mut_path, sep="\t", comment="#")


[INFO] Added 94 passenger (_MUT) features from 7983 uncertain mutations.
[INFO] One-hot matrix: 199 samples × 122 features
  • 17 GOF
  • 11 LOF
  • 94 MUT
[INFO] CNA: mapped 191 samples
[INFO] Fusions: mapped 83 samples
[INFO] Shared sample count: 78
 Mutations: 42 features
 Fusions:   7 features
 CNAs:      200 features
 Clinical:  18 features
 Subtypes:  1 unique
[Gene selection]
  Total RNA genes: 20531
  Altered genes present in RNA: 179
  Target total: 10000 -> selecting 9821 non-altered to fill
  Low-expression (non-altered) genes dropped: 3755
  NOTE: 24 altered gene(s) are lowly expressed but KEPT: DPP6, NOBOX, HYALP1, PPP1R3A, HYAL4, C7orf33, ASB15, PAX4, CNPY1, GALNTL5 ...
[preprocess_rna] Applied scale factor: 10
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 2.55 seconds.

Fitting dispersion trend curve...
... done in 1.35 seconds.

Fitting MAP dispersions...
... done in 3.38 seconds.

Fitting LFCs...
... done in 1.63 seconds.

Calculating cook's distance...
... done in 0.06 seconds.

Replacing 1021 outlier genes.

Fitting dispersions...
... done in 0.29 seconds.

Fitting MAP dispersions...
... done in 0.29 seconds.

Fitting LFCs...
... done in 0.26 seconds.



[✓] Simulation outputs saved for 'AML' in: /home/alanah/Documents
